# TiHon-TTS: quick test

Vietnamese text-to-speech small enough for a microcontroller: 100,118 parameters, four voices. Run the cells in order; everything runs on CPU.

In [ ]:
import sys

from huggingface_hub import snapshot_download

repo = snapshot_download("ngocthien/tihon-tts")
sys.path.insert(0, repo)

In [ ]:
import time

import numpy as np
import torch
from IPython.display import Audio, display

from tihon_tts.model import TiHonTTS
from tihon_tts.text import text_to_ids

SR = 24_000
VOICES = ["ngochuyen", "infore", "trucly", "haidang"]
model = TiHonTTS.from_pretrained(repo).eval()


def tts(text: str, voice: str = "trucly", seed: int = 0) -> np.ndarray:
    """Text → float32 waveform at 24 kHz (0.3 s pause after each chunk)."""
    model.set_postprocessing(voice)
    gen = torch.Generator().manual_seed(seed)
    pause = np.zeros(int(0.3 * SR), dtype=np.float32)
    parts = []
    with torch.no_grad():
        for ids in text_to_ids(text):
            parts += [model(torch.tensor([ids]), voice, gen).numpy(), pause]
    return np.concatenate(parts)

## Your text

In [ ]:
text = "Xin chào, hôm nay trời đẹp không?"  # @param {type:"string"}
voice = "trucly"  # @param ["ngochuyen", "infore", "trucly", "haidang"]

start = time.perf_counter()
wav = tts(text, voice)
print(f"{len(wav) / SR:.2f} s of audio in {time.perf_counter() - start:.2f} s")
display(Audio(wav, rate=SR))

## All four voices

In [ ]:
sentence = "Giá vé là 1.250.000 đồng, giảm 15% nếu mua trước ngày 25/12."
for v in VOICES:
    print(v)
    display(Audio(tts(sentence, v), rate=SR))

## With Transformers

The same model through the `text-to-speech` pipeline of Transformers (`trust_remote_code=True` loads the small
wrapper in this repository).

In [ ]:
from transformers import pipeline

tts_pipe = pipeline("text-to-speech", model="ngocthien/tihon-tts", trust_remote_code=True)
out = tts_pipe(text, voice=voice)
display(Audio(out["audio"], rate=out["sampling_rate"]))